## 1. 配置：路径、窗口、超参数

In [ ]:

RAW_PATH   = 'D:/investment/Data/hs300_factor_data_filled.parquet'
PANEL_PATH = 'D:/investment/Data/factor_panel_v2_processed.parquet'
IC_PATH    = 'D:/investment/Data/03v3_ic_summary.csv'

IC_TSTAT_THRESHOLD = 2.0
MIN_COVERAGE       = 0.70
FWD_RET_COVERAGE   = 0.80
SEED               = 42

TRAIN_WINDOW = 36
STEP_SIZE    = 3

LAMBDA      = 1.0
LAMBDA_LIST = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

GAMMA       = 0.003          # 论文口径：成本项为 0.5 * GAMMA * ||Δw||_1
MAX_W_FRAC  = 10             # 单票上限：10/N

# ===== PPP 优化超参数 =====
THETA_L2       = 1e-4         # shrinkage: 防止 theta 过大，保留可解释性
THETA_BOUND    = 5.0          # theta 边界；过大容易被投影层全部截断
OPT_METHOD     = 'Powell'     # 无需梯度；适合 projected PPP baseline
OPT_MAXITER    = 300
OPT_XTOL       = 1e-5
OPT_FTOL       = 1e-6
USE_PREV_THETA_AS_INIT = False # True 会更平滑；False 每个窗口独立估计，更像独立 baseline

# ===== MSE alignment diagnostic，不参与 PPP 权重训练 =====
RIDGE_MSE_ALPHA = 1e-4

# ===== 是否执行多 lambda 扫描 =====
RUN_BASE_EXPERIMENT = True
RUN_LAMBDA_SWEEP    = True

print("=" * 80)
print("PPP Linear Baseline — aligned with E2E v5c Eq.(21)")
print("=" * 80)
print(f"TRAIN_WINDOW={TRAIN_WINDOW}, STEP_SIZE={STEP_SIZE}")
print(f"LAMBDA={LAMBDA}, LAMBDA_LIST={LAMBDA_LIST}, GAMMA={GAMMA}")
print(f"Loss uses: -w'r + 0.5*gamma*||Δw||_1 + lambda*sqrt(w'Dw)")
print(f"Constraints: long-only, sum(w)=1, max(w)<= {MAX_W_FRAC}/N")
print(f"PPP optimizer: {OPT_METHOD}, maxiter={OPT_MAXITER}, theta_l2={THETA_L2}, theta_bound={THETA_BOUND}")

PPP Linear Baseline — aligned with E2E v5c Eq.(21)
TRAIN_WINDOW=36, STEP_SIZE=3
LAMBDA=1.0, LAMBDA_LIST=[0.1, 0.5, 1.0, 2.0, 5.0, 10.0], GAMMA=0.003
Loss uses: -w'r + 0.5*gamma*||Δw||_1 + lambda*sqrt(w'Dw)
Constraints: long-only, sum(w)=1, max(w)<= 10/N
PPP optimizer: Powell, maxiter=300, theta_l2=0.0001, theta_bound=5.0


## 2. 导入与数据加载

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, time, os
from collections import Counter
from scipy.optimize import minimize

warnings.filterwarnings('ignore')
np.random.seed(SEED)

plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

panel = pd.read_parquet(PANEL_PATH)
panel['date'] = pd.to_datetime(panel['date'])

ic_df = pd.read_csv(IC_PATH, index_col=0)
if 'Sig(|t|>2)' in ic_df.columns:
    sig_mask = ic_df['Sig(|t|>2)']
elif 't_stat' in ic_df.columns:
    sig_mask = ic_df['t_stat'].abs() > IC_TSTAT_THRESHOLD
else:
    raise ValueError("IC 文件需含 t_stat 或 Sig(|t|>2) 列")

SIG_FACTORS = list(ic_df[sig_mask].index)
N_FEATURES  = len(SIG_FACTORS)

print(f"Factor panel: {panel.shape}  ({panel['date'].nunique()} months)")
print(f"显著因子 ({N_FEATURES} 个): {SIG_FACTORS}")
assert N_FEATURES >= 2, "显著因子数量太少，无法稳定估计 PPP policy."

Factor panel: (21760, 105)  (68 months)
显著因子 (10 个): ['F91_ocf_ni', 'F97_accruals', 'F80_ocf_mv', 'F74_cfp', 'F55_holding_adj', 'F81_size', 'F34_vol_of_vol', 'F88_cash_roe', 'F41_turn_std', 'F30_range_vol']


## 3. 股票池、截面标准化与张量/数组构造

In [5]:
all_dates = sorted(panel['date'].unique())

def get_valid_stocks(df, factors):
    return set(df.dropna(subset=factors + ['fwd_ret'])['ts_code'].unique())

month_valid = {d: get_valid_stocks(panel[panel['date'] == d], SIG_FACTORS) for d in all_dates}
stock_cnt   = Counter(s for v in month_valid.values() for s in v)
universe    = sorted(s for s, c in stock_cnt.items() if c / len(all_dates) >= MIN_COVERAGE)

N = len(universe)
MAX_W = MAX_W_FRAC / N
EW = np.ones(N, dtype=np.float64) / N

panel_u = (
    panel[panel['ts_code'].isin(universe)]
    .sort_values(['date', 'ts_code'])
    .reset_index(drop=True)
)

# 与 E2E v5c notebook 保持相同的缺失值处理方式
for col in SIG_FACTORS:
    panel_u[col] = panel_u.groupby('date')[col].transform(
        lambda x: x.fillna(0 if x.isna().all() else x.mean())
    )

panel_u['fwd_ret'] = panel_u.groupby('date')['fwd_ret'].transform(
    lambda x: x.fillna(x.median() if not x.isna().all() else 0)
)

def cross_sec_zscore(arr):
    arr = np.asarray(arr, dtype=np.float64)
    mu  = arr.mean(axis=0, keepdims=True)
    sig = np.maximum(arr.std(axis=0, keepdims=True), 1e-8)
    return np.nan_to_num((arr - mu) / sig).astype(np.float64)

fwd_cov = panel_u.groupby('date')['fwd_ret'].apply(lambda x: x.notna().mean())
dates_valid = [d for d in sorted(panel_u['date'].unique()) if fwd_cov.get(d, 0) >= FWD_RET_COVERAGE]

X_arrays, y_arrays = [], []
for d in dates_valid:
    sub = panel_u[panel_u['date'] == d].sort_values('ts_code')
    # 检查每个月股票顺序是否完整一致
    if list(sub['ts_code']) != universe:
        # 如果某月有缺失，按 universe 重新索引并填充；通常不会触发
        sub = sub.set_index('ts_code').reindex(universe).reset_index()
        for col in SIG_FACTORS:
            sub[col] = sub[col].fillna(sub[col].mean())
        sub['fwd_ret'] = sub['fwd_ret'].fillna(sub['fwd_ret'].median())
    X_arrays.append(cross_sec_zscore(sub[SIG_FACTORS].values))
    y_arrays.append(sub['fwd_ret'].values.astype(np.float64))

T = len(X_arrays)
print(f"股票池: {N} 只")
print(f"单票上限 MAX_W = {MAX_W_FRAC}/N = {MAX_W:.6f}")
print(f"有效月份: {T}")
print(f"首月/末月: {dates_valid[0].date()} / {dates_valid[-1].date()}")

股票池: 252 只
单票上限 MAX_W = 10/N = 0.039683
有效月份: 68
首月/末月: 2020-04-30 / 2025-11-28


## 4. 滚动窗口：36 个月训练，3 个月测试

In [6]:
rolling_windows = []
start = 0
while start + TRAIN_WINDOW + STEP_SIZE <= T:
    rolling_windows.append({
        'train_start': start,
        'train_end':   start + TRAIN_WINDOW,
        'test_start':  start + TRAIN_WINDOW,
        'test_end':    min(start + TRAIN_WINDOW + STEP_SIZE, T)
    })
    start += STEP_SIZE

assert rolling_windows, "有效月份不足，无法构造 rolling windows."

fw, lw = rolling_windows[0], rolling_windows[-1]
print(f"滚动窗口: {len(rolling_windows)} 个")
print(f"首: [{dates_valid[fw['train_start']].date()} ~ {dates_valid[fw['train_end']-1].date()}] "
      f"→ [{dates_valid[fw['test_start']].date()} ~ {dates_valid[fw['test_end']-1].date()}]")
print(f"末: 测试 [{dates_valid[lw['test_start']].date()} ~ {dates_valid[lw['test_end']-1].date()}]")

滚动窗口: 10 个
首: [2020-04-30 ~ 2023-03-31] → [2023-04-28 ~ 2023-06-30]
末: 测试 [2025-07-31 ~ 2025-09-30]


## 5. PPP 权重函数与 capped simplex 投影

原始 PPP 线性 policy：

\[
\tilde w_{i,t}=\bar w_{i,t}+\frac{1}{N}\theta^\top \hat x_{i,t}.
\]

为了与 E2E v5c 使用完全相同的可行域，本 notebook 将 \(\tilde w_t\) 投影到 capped simplex：

\[
\mathcal W=\left\{w: w_i\ge 0,\ \sum_i w_i=1,\ w_i\le 10/N\right\}.
\]

这一步是 baseline 对齐所必需的：否则原始 PPP 会允许 long-short，而 E2E v5c 是 long-only + 单票上限。

In [7]:
def project_capped_simplex(v, z=1.0, u=None, tol=1e-12, max_iter=100):
    """
    Euclidean projection onto capped simplex:
        {w: sum(w)=z, 0 <= w_i <= u}

    Solution has form:
        w_i = clip(v_i - tau, 0, u)
    where tau is found by bisection.

    This is deterministic and does not require cvxpy.
    """
    v = np.asarray(v, dtype=np.float64)
    n = v.size
    if u is None:
        u = z
    if n * u < z - 1e-12:
        raise ValueError(f"Infeasible capped simplex: n*u={n*u:.6f} < z={z:.6f}")

    # If v is already feasible, return it after tiny normalization correction
    if (v.min() >= -tol) and (v.max() <= u + tol) and (abs(v.sum() - z) <= 1e-10):
        w = np.clip(v, 0.0, u)
        return w / w.sum() * z

    lo = np.min(v - u) - 1.0
    hi = np.max(v) + 1.0

    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        w = np.clip(v - mid, 0.0, u)
        if w.sum() > z:
            lo = mid
        else:
            hi = mid
        if abs(w.sum() - z) < tol:
            break

    tau = 0.5 * (lo + hi)
    w = np.clip(v - tau, 0.0, u)

    # Numerical correction: distribute tiny residual over non-bound coordinates
    residual = z - w.sum()
    if abs(residual) > 1e-10:
        free = (w > 1e-10) & (w < u - 1e-10)
        if free.any():
            w[free] += residual / free.sum()
            w = np.clip(w, 0.0, u)
        else:
            # Fallback: renormalize then clip; should rarely be needed
            s = w.sum()
            if s > 0:
                w = np.clip(w / s * z, 0.0, u)

    # Final small correction if still needed
    if abs(w.sum() - z) > 1e-8:
        # Put remaining residual into the largest non-capped position
        residual = z - w.sum()
        candidates = np.where(w < u - 1e-8)[0] if residual > 0 else np.where(w > 1e-8)[0]
        if len(candidates) > 0:
            j = candidates[np.argmax(np.abs(w[candidates]))]
            w[j] = np.clip(w[j] + residual, 0.0, u)

    return w.astype(np.float64)


def ppp_raw_weight(theta, X, benchmark=None):
    """
    Brandt-Santa-Clara-Valkanov linear PPP:

        raw_w = benchmark + (X @ theta) / N

    X is cross-sectionally standardized N x K matrix.
    benchmark defaults to equal weight because the existing E2E notebook benchmarks against EW.
    """
    X = np.asarray(X, dtype=np.float64)
    theta = np.asarray(theta, dtype=np.float64)
    n = X.shape[0]
    if benchmark is None:
        benchmark = np.ones(n, dtype=np.float64) / n
    return benchmark + (X @ theta) / n


def ppp_weight(theta, X, benchmark=None, max_w=MAX_W):
    """
    Feasible PPP weight:
        raw PPP tilt + projection to {long-only, fully invested, capped}.
    """
    raw = ppp_raw_weight(theta, X, benchmark=benchmark)
    return project_capped_simplex(raw, z=1.0, u=max_w)


# Sanity check
_v = np.random.randn(N)
_w = project_capped_simplex(_v, z=1.0, u=MAX_W)
print("Projection sanity check:")
print(f"sum(w)={_w.sum():.12f}, min(w)={_w.min():.6e}, max(w)={_w.max():.6f}, MAX_W={MAX_W:.6f}")

_theta0 = np.zeros(N_FEATURES)
_w0 = ppp_weight(_theta0, X_arrays[0])
print("PPP theta=0 check:")
print(f"sum(w0)={_w0.sum():.12f}, max(w0)={_w0.max():.6f}, distance_to_EW={np.abs(_w0 - EW).sum():.4e}")

Projection sanity check:
sum(w)=1.000000000000, min(w)=0.000000e+00, max(w)=0.039683, MAX_W=0.039683
PPP theta=0 check:
sum(w0)=1.000000000000, max(w0)=0.003968, distance_to_EW=2.1858e-16


## 6. 训练目标与 MSE alignment diagnostic

PPP 训练目标：

\[
\frac1T\sum_t
\left[
-w_t^\top r_{t+1}
+\frac{\gamma}{2}\|w_t-w_{t-1}\|_1
+\lambda\sqrt{w_t^\top D w_t}
\right]
+\rho\|\theta\|_2^2.
\]

其中 \(D\) 使用训练窗口内每只股票的时间序列方差，保持和 E2E notebook 中的 diagonal risk vector 对齐。

额外 MSE diagnostic：对每个 rolling window，用同样的 \(X_t,r_{t+1}\) 拟合线性 Ridge return predictor，只用于核查数据、窗口和收益尺度，不进入 PPP objective。

In [8]:
def compute_var_vec(y_list):
    mat = np.stack(y_list, axis=0)
    return np.maximum(mat.var(axis=0), 1e-6).astype(np.float64)


def ppp_training_objective(theta, X_tr, y_tr, var_vec, lambda_, gamma,
                           theta_l2=THETA_L2, max_w=MAX_W, benchmark=None,
                           return_details=False):
    """
    Sequential PPP objective over a training window.

    w_prev is reset to equal weight inside every objective evaluation, matching the
    chronological dependence used in E2E training but avoiding backprop through time.
    """
    theta = np.asarray(theta, dtype=np.float64)
    w_prev = EW.copy() if benchmark is None else np.asarray(benchmark, dtype=np.float64).copy()

    losses, gross_list, tc_list, risk_list = [], [], [], []
    for X_t, y_t in zip(X_tr, y_tr):
        w = ppp_weight(theta, X_t, benchmark=benchmark, max_w=max_w)

        gross = float(np.dot(w, y_t))
        tc    = 0.5 * gamma * float(np.abs(w - w_prev).sum())
        risk  = lambda_ * float(np.sqrt(np.sum((w ** 2) * var_vec) + 1e-12))

        losses.append(-gross + tc + risk)
        gross_list.append(gross)
        tc_list.append(tc)
        risk_list.append(risk)

        w_prev = w

    obj = float(np.mean(losses) + theta_l2 * np.dot(theta, theta))

    if not np.isfinite(obj):
        obj = 1e6

    if return_details:
        return {
            'objective': obj,
            'gross_mean': float(np.mean(gross_list)),
            'tc_mean': float(np.mean(tc_list)),
            'risk_mean': float(np.mean(risk_list)),
            'loss_mean_without_l2': float(np.mean(losses)),
            'theta_l2_term': float(theta_l2 * np.dot(theta, theta))
        }
    return obj


def fit_mse_alignment_diagnostic(X_tr, y_tr, X_te, y_te, ridge_alpha=RIDGE_MSE_ALPHA):
    """
    Simple Ridge linear return predictor:
        r_hat = a + x'beta

    This is NOT used by PPP. It only reports MSE so that this notebook can be
    checked against the E2E notebook's data/window alignment.
    """
    Xtr = np.vstack(X_tr)
    ytr = np.concatenate(y_tr)
    Xte = np.vstack(X_te)
    yte = np.concatenate(y_te)

    Ztr = np.column_stack([np.ones(len(Xtr)), Xtr])
    Zte = np.column_stack([np.ones(len(Xte)), Xte])

    k = Ztr.shape[1]
    P = np.eye(k)
    P[0, 0] = 0.0  # do not penalize intercept

    beta = np.linalg.solve(Ztr.T @ Ztr + ridge_alpha * P, Ztr.T @ ytr)

    pred_tr = Ztr @ beta
    pred_te = Zte @ beta

    return {
        'mse_train': float(np.mean((pred_tr - ytr) ** 2)),
        'mse_test':  float(np.mean((pred_te - yte) ** 2)),
        'beta': beta
    }


def optimize_ppp_theta(X_tr, y_tr, var_vec, lambda_, gamma, theta0=None,
                       theta_l2=THETA_L2, theta_bound=THETA_BOUND,
                       opt_method=OPT_METHOD, maxiter=OPT_MAXITER):
    if theta0 is None:
        theta0 = np.zeros(N_FEATURES, dtype=np.float64)
    else:
        theta0 = np.asarray(theta0, dtype=np.float64)

    bounds = [(-theta_bound, theta_bound)] * N_FEATURES

    def obj(th):
        return ppp_training_objective(
            th, X_tr, y_tr, var_vec, lambda_=lambda_, gamma=gamma,
            theta_l2=theta_l2, max_w=MAX_W
        )

    t0 = time.time()
    result = minimize(
        obj,
        theta0,
        method=opt_method,
        bounds=bounds if opt_method in ['Powell', 'L-BFGS-B', 'TNC', 'SLSQP'] else None,
        options={
            'maxiter': maxiter,
            'xtol': OPT_XTOL,
            'ftol': OPT_FTOL,
            'disp': False
        }
    )

    # Fallback if Powell fails badly
    if (not result.success) or (not np.isfinite(result.fun)):
        result2 = minimize(
            obj,
            np.clip(theta0, -theta_bound, theta_bound),
            method='L-BFGS-B',
            bounds=bounds,
            options={'maxiter': maxiter, 'ftol': OPT_FTOL, 'disp': False}
        )
        if np.isfinite(result2.fun) and result2.fun < result.fun:
            result = result2

    theta_hat = np.clip(np.asarray(result.x, dtype=np.float64), -theta_bound, theta_bound)
    details = ppp_training_objective(
        theta_hat, X_tr, y_tr, var_vec, lambda_=lambda_, gamma=gamma,
        theta_l2=theta_l2, max_w=MAX_W, return_details=True
    )

    info = {
        'success': bool(result.success),
        'message': str(result.message),
        'fun': float(result.fun) if np.isfinite(result.fun) else np.nan,
        'nfev': int(getattr(result, 'nfev', -1)),
        'nit': int(getattr(result, 'nit', -1)),
        'seconds': float(time.time() - t0),
        **details
    }

    return theta_hat, info

print("PPP objective, optimizer, and MSE diagnostic ready.")

PPP objective, optimizer, and MSE diagnostic ready.


## 7. 滚动训练与测试

In [9]:
def run_rolling_ppp(lambda_=LAMBDA, gamma=GAMMA, label=None):
    """
    Rolling PPP baseline.

    For each window:
      1. Estimate theta on the 36-month training window by PPP utility objective.
      2. Apply this theta to the next 3 months.
      3. Evaluate realized net returns using the same TC convention:
             net = gross - 0.5 * gamma * ||w_t - w_{t-1}||_1
      4. Record MSE diagnostic from a linear Ridge return model on the same window.
    """
    if label is None:
        label = f"ppp_lam{lambda_}_gamma{gamma}"

    res = {
        'returns': [], 'gross_returns': [], 'tc_costs': [], 'risk_terms': [],
        'weights': [], 'dates': [], 'bench': [],
        'theta_by_window': [], 'theta_dates': [],
        'opt_info': [], 'mse_train': [], 'mse_test': []
    }

    w_prev_test = EW.copy()
    prev_theta = np.zeros(N_FEATURES, dtype=np.float64)
    t_start = time.time()

    for idx, win in enumerate(rolling_windows):
        ts, te = win['train_start'], win['train_end']
        vs, ve = win['test_start'],  win['test_end']

        X_tr = X_arrays[ts:te]
        y_tr = y_arrays[ts:te]
        X_te = X_arrays[vs:ve]
        y_te = y_arrays[vs:ve]

        var_vec = compute_var_vec(y_tr)

        theta0 = prev_theta if USE_PREV_THETA_AS_INIT else np.zeros(N_FEATURES, dtype=np.float64)

        theta_hat, info = optimize_ppp_theta(
            X_tr, y_tr, var_vec,
            lambda_=lambda_, gamma=gamma,
            theta0=theta0,
            theta_l2=THETA_L2,
            theta_bound=THETA_BOUND,
            opt_method=OPT_METHOD,
            maxiter=OPT_MAXITER
        )

        prev_theta = theta_hat.copy()

        mse_info = fit_mse_alignment_diagnostic(X_tr, y_tr, X_te, y_te)
        res['mse_train'].append(mse_info['mse_train'])
        res['mse_test'].append(mse_info['mse_test'])

        res['theta_by_window'].append(theta_hat)
        res['theta_dates'].append(dates_valid[te-1])
        res['opt_info'].append(info)

        for i in range(vs, ve):
            w = ppp_weight(theta_hat, X_arrays[i], max_w=MAX_W)

            gross_ret = float(np.dot(w, y_arrays[i]))
            tc_cost   = 0.5 * gamma * float(np.abs(w - w_prev_test).sum())
            net_ret   = gross_ret - tc_cost
            risk_term = lambda_ * float(np.sqrt(np.sum((w ** 2) * var_vec) + 1e-12))

            res['returns'].append(net_ret)
            res['gross_returns'].append(gross_ret)
            res['tc_costs'].append(tc_cost)
            res['risk_terms'].append(risk_term)
            res['weights'].append(w.copy())
            res['dates'].append(dates_valid[i])
            res['bench'].append(float(np.mean(y_arrays[i])))

            w_prev_test = w.copy()

        elapsed = (time.time() - t_start) / 60
        theta_norm = float(np.linalg.norm(theta_hat))
        print(
            f"[PPP | lambda={lambda_:>4} | gamma={gamma}] "
            f"win {idx+1:02d}/{len(rolling_windows)}  "
            f"[{dates_valid[ts].strftime('%y-%m')}~{dates_valid[te-1].strftime('%y-%m')}]"
            f" -> [{dates_valid[vs].strftime('%y-%m')}~{dates_valid[ve-1].strftime('%y-%m')}]  "
            f"obj={info['loss_mean_without_l2']:+.5f}  "
            f"|theta|={theta_norm:.3f}  "
            f"mse_te={mse_info['mse_test']:.6g}  "
            f"nfev={info['nfev']}  {elapsed:.1f}min"
        )

    return res


def calc_turnover(weights_list):
    to = []
    w_prev = EW.copy()
    for w in weights_list:
        to.append(float(np.abs(w - w_prev).sum()))  # L1 = two-sided turnover
        w_prev = np.asarray(w, dtype=np.float64)
    return np.array(to, dtype=np.float64)


def metrics(returns, weights_list=None, label='', verbose=True):
    r = np.asarray(returns, dtype=np.float64)
    ann_ret = r.mean() * 12
    ann_vol = r.std(ddof=1) * np.sqrt(12) if len(r) > 1 else 0.0
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else np.nan

    cum = np.cumprod(1 + r)
    dd = (cum / np.maximum.accumulate(cum)) - 1

    neg = r[r < 0]
    sortino = ann_ret / (neg.std(ddof=1) * np.sqrt(12)) if len(neg) > 1 and neg.std(ddof=1) > 0 else np.nan
    win = float((r > 0).mean())

    if weights_list is not None:
        to = calc_turnover(weights_list)
        avg_to = float(to.mean())
    else:
        to = np.array([0.0])
        avg_to = 0.0

    out = dict(
        Ann_Ret=float(ann_ret), Ann_Vol=float(ann_vol), Sharpe=float(sharpe),
        Sortino=float(sortino), MDD=float(dd.min()), Win=win, AvgTO=avg_to,
        Mean=float(r.mean()), Std=float(r.std(ddof=1)) if len(r) > 1 else 0.0
    )

    if verbose:
        print(
            f"{label:<24} AnnRet={ann_ret:+.2%}  AnnVol={ann_vol:.2%}  "
            f"SR={sharpe:.3f}  Sortino={sortino:.3f}  MDD={dd.min():.2%}  "
            f"Win={win:.1%}  AvgTO(two-sided)={avg_to:.2%}"
        )
    return out


def save_ppp_outputs(res, prefix):
    dates_idx = [d.strftime('%Y-%m') for d in res['dates']]
    theta_idx = [d.strftime('%Y-%m') for d in res['theta_dates']]

    weights_df = pd.DataFrame(np.array(res['weights']), index=dates_idx, columns=universe)
    theta_df = pd.DataFrame(np.array(res['theta_by_window']), index=theta_idx, columns=SIG_FACTORS)

    perf_df = pd.DataFrame({
        'date': dates_idx,
        'net_return': res['returns'],
        'gross_return': res['gross_returns'],
        'tc_cost': res['tc_costs'],
        'risk_term_trainD': res['risk_terms'],
        'bench_ew_return': res['bench']
    })

    mse_df = pd.DataFrame({
        'window_end': theta_idx,
        'mse_train_diag': res['mse_train'],
        'mse_test_diag': res['mse_test'],
    })

    weights_path = f'./weights_{prefix}.csv'
    theta_path   = f'./theta_{prefix}.csv'
    perf_path    = f'./perf_{prefix}.csv'
    mse_path     = f'./mse_diag_{prefix}.csv'

    weights_df.to_csv(weights_path, encoding='utf-8-sig')
    theta_df.to_csv(theta_path, encoding='utf-8-sig')
    perf_df.to_csv(perf_path, encoding='utf-8-sig', index=False)
    mse_df.to_csv(mse_path, encoding='utf-8-sig', index=False)

    print(f"Saved: {weights_path}")
    print(f"Saved: {theta_path}")
    print(f"Saved: {perf_path}")
    print(f"Saved: {mse_path}")

    return {
        'weights_path': weights_path,
        'theta_path': theta_path,
        'perf_path': perf_path,
        'mse_path': mse_path
    }

print("Rolling PPP runner and metrics ready.")

Rolling PPP runner and metrics ready.


## 8. 单个 lambda 基准实验

In [10]:
if RUN_BASE_EXPERIMENT:
    print("=" * 86)
    print(f"PPP baseline  (lambda={LAMBDA}, gamma={GAMMA}, realized cost=0.5*gamma*||Δw||_1)")
    print("=" * 86)

    res_ppp = run_rolling_ppp(lambda_=LAMBDA, gamma=GAMMA)

    r_ppp   = np.array(res_ppp['returns'])
    r_bench = np.array(res_ppp['bench'])

    print("\n" + "=" * 86)
    print(f"Test-period results  (lambda={LAMBDA}, gamma={GAMMA})")
    print("=" * 86)

    m_ppp   = metrics(r_ppp,   res_ppp['weights'], label=f'PPP linear λ={LAMBDA}')
    m_bench = metrics(r_bench, None,               label='Equal-Weight')

    print("\nMSE alignment diagnostic:")
    print(f"Mean train MSE = {np.mean(res_ppp['mse_train']):.8f}")
    print(f"Mean test  MSE = {np.mean(res_ppp['mse_test']):.8f}")
    print("说明：该 MSE 由同窗口 Ridge 线性收益预测产生，不参与 PPP 权重优化，仅用于和 E2E notebook 的数据窗口/收益尺度对照。")

    save_ppp_outputs(res_ppp, prefix=f"ppp_linear_lam{LAMBDA}_gamma{GAMMA}".replace('.', 'p'))
else:
    print("RUN_BASE_EXPERIMENT=False, skipped.")

PPP baseline  (lambda=1.0, gamma=0.003, realized cost=0.5*gamma*||Δw||_1)
[PPP | lambda= 1.0 | gamma=0.003] win 01/10  [20-04~23-03] -> [23-04~23-06]  obj=-0.02403  |theta|=4.827  mse_te=0.0115812  nfev=8321  1.4min
[PPP | lambda= 1.0 | gamma=0.003] win 02/10  [20-07~23-06] -> [23-07~23-09]  obj=-0.01326  |theta|=3.787  mse_te=0.00738033  nfev=6781  2.5min
[PPP | lambda= 1.0 | gamma=0.003] win 03/10  [20-10~23-09] -> [23-10~23-12]  obj=-0.01086  |theta|=3.820  mse_te=0.011652  nfev=5833  3.5min
[PPP | lambda= 1.0 | gamma=0.003] win 04/10  [21-01~23-12] -> [24-01~24-03]  obj=-0.00394  |theta|=3.892  mse_te=0.0137999  nfev=5032  4.3min
[PPP | lambda= 1.0 | gamma=0.003] win 05/10  [21-04~24-03] -> [24-04~24-06]  obj=-0.00659  |theta|=3.200  mse_te=0.00826275  nfev=8974  5.8min
[PPP | lambda= 1.0 | gamma=0.003] win 06/10  [21-07~24-06] -> [24-07~24-09]  obj=-0.00592  |theta|=3.717  mse_te=0.0306574  nfev=3888  6.4min
[PPP | lambda= 1.0 | gamma=0.003] win 07/10  [21-10~24-09] -> [24-10~24-1

## 9. 多 lambda 扫描

In [ ]:
if RUN_LAMBDA_SWEEP:
    print("=" * 86)
    print(f"PPP multi-lambda sweep at gamma={GAMMA}: {LAMBDA_LIST}")
    print("=" * 86)

    sweep_results = {}
    sweep_rows = []

    for lam in LAMBDA_LIST:
        print("\n" + "-" * 72)
        print(f"lambda = {lam}")
        print("-" * 72)

        _res = run_rolling_ppp(lambda_=lam, gamma=GAMMA)
        _m = metrics(np.array(_res['returns']), _res['weights'], label=f'PPP λ={lam}', verbose=True)

        _row = {
            'lambda': lam,
            **_m,
            'MeanTrainMSE_diag': float(np.mean(_res['mse_train'])),
            'MeanTestMSE_diag':  float(np.mean(_res['mse_test'])),
            'MeanThetaNorm': float(np.mean([np.linalg.norm(th) for th in _res['theta_by_window']])),
            'MeanTCostMonthly': float(np.mean(_res['tc_costs'])),
            'MeanGrossMonthly': float(np.mean(_res['gross_returns'])),
        }

        sweep_results[lam] = {'res': _res, 'metrics': _m, 'row': _row}
        sweep_rows.append(_row)

        save_ppp_outputs(_res, prefix=f"ppp_linear_lam{lam}_gamma{GAMMA}".replace('.', 'p'))

    sweep_df = pd.DataFrame(sweep_rows)
    sweep_df.to_csv('./ppp_lambda_sweep_summary.csv', encoding='utf-8-sig', index=False)

    print("\n" + "=" * 86)
    print("PPP Lambda Sweep Summary")
    print("=" * 86)
    display_cols = ['lambda', 'Ann_Ret', 'Ann_Vol', 'Sharpe', 'Sortino', 'MDD', 'AvgTO',
                    'MeanTestMSE_diag', 'MeanThetaNorm', 'MeanTCostMonthly']
    print(sweep_df[display_cols].to_string(index=False))
    print("\nSaved: ./ppp_lambda_sweep_summary.csv")
else:
    print("RUN_LAMBDA_SWEEP=False, skipped.")

## 10. 可视化

In [ ]:
# 如果已经运行 base experiment，则画 PPP vs Equal Weight
if 'res_ppp' in globals():
    r_ppp   = np.array(res_ppp['returns'])
    r_bench = np.array(res_ppp['bench'])

    N_TEST  = len(r_ppp)
    x       = np.arange(N_TEST)
    dlabels = [d.strftime('%Y-%m') for d in res_ppp['dates']]
    step    = max(1, N_TEST // 8)

    cum_ppp = np.cumprod(1 + r_ppp)
    cum_bch = np.cumprod(1 + r_bench)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    ax = axes[0, 0]
    ax.plot(cum_ppp, lw=2.2, label=f"PPP λ={LAMBDA}  SR={m_ppp['Sharpe']:.2f}  Ann={m_ppp['Ann_Ret']:.1%}")
    ax.plot(cum_bch, lw=1.6, ls='--', label=f"Equal Weight  SR={m_bench['Sharpe']:.2f}")
    ax.axhline(1, lw=0.5)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_title(f'Cumulative return: PPP vs EW  |  γ={GAMMA}')
    ax.set_xticks(x[::step])
    ax.set_xticklabels(dlabels[::step], rotation=30, ha='right', fontsize=7)

    def drawdown(cum):
        return (cum / np.maximum.accumulate(cum)) - 1

    ax = axes[0, 1]
    ax.fill_between(x, 0, drawdown(cum_ppp), alpha=0.35, label='PPP')
    ax.fill_between(x, 0, drawdown(cum_bch), alpha=0.25, label='EW')
    ax.set_title('Drawdown')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xticks(x[::step])
    ax.set_xticklabels(dlabels[::step], rotation=30, ha='right', fontsize=7)

    ax = axes[1, 0]
    to_ppp = calc_turnover(res_ppp['weights'])
    ax.plot(to_ppp * 100, lw=1.8, label=f'PPP avg={to_ppp.mean():.1%}')
    ax.set_title('Two-sided turnover (%)')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xticks(x[::step])
    ax.set_xticklabels(dlabels[::step], rotation=30, ha='right', fontsize=7)

    ax = axes[1, 1]
    theta_df_plot = pd.DataFrame(np.array(res_ppp['theta_by_window']), columns=SIG_FACTORS)
    for col in theta_df_plot.columns:
        ax.plot(theta_df_plot[col].values, lw=1.2, label=col)
    ax.axhline(0, lw=0.5)
    ax.set_title('PPP policy coefficients by rolling window')
    ax.legend(fontsize=7, ncol=2)
    ax.grid(alpha=0.3)
    ax.set_xlabel('Rolling window index')

    plt.suptitle(f'PPP Linear Baseline  |  λ={LAMBDA}  γ={GAMMA}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()


# 如果已经运行 lambda sweep，则画 realized frontier
if 'sweep_results' in globals() and len(sweep_results) > 0:
    lam_vals = [l for l in LAMBDA_LIST if l in sweep_results]
    vols = [sweep_results[l]['metrics']['Ann_Vol'] for l in lam_vals]
    rets = [sweep_results[l]['metrics']['Ann_Ret'] for l in lam_vals]
    srs  = [sweep_results[l]['metrics']['Sharpe']  for l in lam_vals]
    tos  = [sweep_results[l]['metrics']['AvgTO']   for l in lam_vals]

    fig2, axes2 = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes2[0]
    ax.scatter(vols, rets, s=80, zorder=4, label='PPP')
    ax.plot(vols, rets, lw=1.5, alpha=0.6)
    for l, v, r in zip(lam_vals, vols, rets):
        ax.annotate(f'λ={l}', (v, r), textcoords='offset points', xytext=(5, 3), fontsize=8)
    ax.set_xlabel('Annualized volatility')
    ax.set_ylabel('Annualized return')
    ax.set_title('PPP realized frontier')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))

    ax = axes2[1]
    ln1 = ax.plot(lam_vals, srs, lw=2, marker='o', label='PPP Sharpe')
    ax2b = ax.twinx()
    ln2 = ax2b.plot(lam_vals, [t * 100 for t in tos], lw=2, ls='--', marker='s', label='TO%')
    ax.set_xlabel('lambda')
    ax.set_ylabel('Sharpe')
    ax2b.set_ylabel('Two-sided turnover (%)')
    ax.set_title('Sharpe & turnover vs lambda')
    ax.grid(alpha=0.3)
    ax.set_xscale('log')
    lns = ln1 + ln2
    ax.legend(lns, [l.get_label() for l in lns], fontsize=9)

    plt.suptitle(f'PPP lambda sweep  |  gamma={GAMMA}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 11. 与 E2E v5c 对齐检查清单

运行后建议检查：

1. `dates_valid`、`rolling_windows` 是否与 `e2e_v5c_eq21_aligned.ipynb` 打印一致。
2. `N`、`SIG_FACTORS` 是否一致。
3. `MeanTestMSE_diag` 与 E2E notebook 中同类 MSE 量级是否接近。  
   注意：PPP 的 MSE diagnostic 是线性 Ridge，不是 PPP objective，也不是 MLP；它主要用来检查数据窗口和收益尺度。
4. `weights_ppp_linear_lam*_gamma*.csv` 的日期索引是否与 E2E 输出权重文件一致。
5. 业绩指标统一使用：
   - 年化收益；
   - 年化波动；
   - Sharpe；
   - Sortino；
   - 最大回撤；
   - 双边月换手率；
   - realized net return = gross return − \(0.5\gamma\|\Delta w\|_1\)。